# Pipeline Sample Snippets — Worked Demo

Six short, illustrative functions from `sample_snippets.py`, walking the
same cascade documented across `../docs/` — type of seating
(contractual / structural-split / prepayment), type of seat (asset /
liability), and the resulting charge, credit, and realised margin.

These are simplified re-implementations of real pipeline functions, not
the production code itself (see `PIPELINE_ARCHITECTURE.md`) — but the
worked examples below reproduce the exact TERM001 / CALL001 / HOME001
numbers used throughout this repo's docs and the top-level README, so
you can watch the same figures actually get computed.

All the logic lives in **`sample_snippets.py`** (imported below) — this
notebook only runs it and displays the results. To change how a
function works, edit that file, not this notebook.

In [ ]:
import numpy as np
import pandas as pd

from sample_snippets import (
    contractual_wal,
    derive_wal_type,
    derive_asset_liability,
    expand_structural_split_tranches,
    lookup_prepayment_wal,
    compute_ftp_charges,
)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Contractual — the account speaks for itself

`TERM001` is a fixed-term deposit. Its ALCO class isn't in the WAL
routing table below at all, so `derive_wal_type` falls back to
`CONTRACTUAL` — the most conservative default. `contractual_wal` then
just passes the real tenor straight through, no adjustment.

In [ ]:
wal_rules = pd.DataFrame({
    "ALCO_CLASS": ["CALL_LIAB", "HOME_LOAN"],
    "WAL_TYPE": ["STRUCTURAL SPLIT", "PREPAYMENT"],
})

term001_class = pd.Series(["TERM_DEPOSIT"])  # not in wal_rules -> defaults to CONTRACTUAL
wal_type = derive_wal_type(term001_class, wal_rules).iloc[0]

term001_wal = contractual_wal(pd.Series([2.00]), pd.Series([np.nan])).iloc[0]

pd.DataFrame([{"ACC_NO": "TERM001", "WAL_TYPE": wal_type, "CALCULATED_WAL": term001_wal}])

## 2. Structural split — one account, four seats at once

`CALL001` is a call account — a liability, R500,000 balance, no
maturity date. `derive_asset_liability` reads its GL line to confirm
the cabin (liability, since the line doesn't start with `'1'`), then
`expand_structural_split_tranches` splits the balance into four seats
on the curve, each with its own share of the balance and its own WAL.

In [ ]:
call001 = pd.Series({"ACC_NO": "CALL001", "OUT_BAL": 500_000.0, "TYPE": "LIABILITY"})

cabin = derive_asset_liability(pd.Series(["2001"]))[0]
print(f"GL line '2001' -> {cabin} (doesn't start with '1')")

split_pct = pd.Series({
    "OVERNIGHT_PCT": 0.18, "SHORT_PCT": 0.22, "MEDIUM_PCT": 0.35, "LONG_PCT": 0.25,
})
tenor_class_wal = pd.DataFrame({
    "TENOR_CLASSIFICATION": ["OVERNIGHT", "SHORT TERM", "MEDIUM TERM", "LONG TERM"],
    "TYPE": ["LIABILITY"] * 4,
    "FINAL_WAL": [0.0027, 0.62, 1.85, 4.20],
})

expand_structural_split_tranches(call001, split_pct, tenor_class_wal)

## 3. Prepayment — a seat closer in than the paperwork says

`HOME001` is a home loan — an asset, 12.4-year contractual maturity.
`lookup_prepayment_wal` finds an exact `AC_CAT` match in the reference
table and returns its adjusted figure directly (tier 1); if it hadn't
matched, tier 2 would have averaged every `AC_CAT` sharing the same
`ALCO_CLASS` instead of falling through to the raw contractual tenor.

In [ ]:
prepayment_ref = pd.DataFrame({
    "AC_CAT": ["HOME_LOAN_STD"],
    "ALCO_CLASS": ["HOME_LOAN"],
    "FINAL_PREPAYMENT_ADJUSTED_WAL": [7.85],
})

home001_wal = lookup_prepayment_wal("HOME_LOAN_STD", "HOME_LOAN", prepayment_ref)

pd.DataFrame([{
    "ACC_NO": "HOME001",
    "CURRENT_AVG_CONTRACTUAL_WAL": 12.4,
    "CALCULATED_WAL": home001_wal,
}])

## 4. Charging the seat — and crediting it

`HOME001`'s seat on the curve (7.85 years) gives it a base rate and
liquidity premium; `compute_ftp_charges` turns that into a daily charge
per component, plus the realised margin against its own observed
customer rate. Because `HOME001` is an **asset**, every charge figure
below is money flowing out of the branch that booked it, to Treasury —
the mirror-image liability example (a credit, not a charge) is in the
README's "Walking one account through" section.

In [ ]:
home001 = pd.DataFrame({
    "ACC_NO": ["HOME001"],
    "TRANCHE_OUT_BAL": [2_000_000.0],
    "BASE_RATE": [0.068],
    "LP": [0.011],
    "OBS_RATE": [0.095],
})

compute_ftp_charges(home001).round(4)